# 🏙️ Next Hot Neighborhood Predictor - Data Collection

## Project Overview

Welcome to Phase 1: Collecting lifestyle amenity data for Austin, TX!

## Step 1: Import Libraries

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from pathlib import Path

# API and Web Scraping
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")
print(f"📅 Notebook run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 2: Configure City (Austin, TX)

In [ ]:
# Austin Configuration
CITY_NAME = "Austin, TX"
CITY_COORDS = (30.2672, -97.7431)
CITY_POPULATION = 978908

print(f"🎯 Target City: {CITY_NAME}")
print(f"📍 Coordinates: {CITY_COORDS}")
print(f"👥 Population: {CITY_POPULATION:,}")

## Step 3: Verify API Key

In [ ]:
# Load API key
YELP_API_KEY = os.getenv('YELP_API_KEY')

if YELP_API_KEY:
    print(f"✅ Yelp API key loaded (length: {len(YELP_API_KEY)})")
    print("🚀 Ready to collect data!")
else:
    print("❌ API key not found! Check your .env file")

## Step 4: Define Yelp API Functions

In [ ]:
import time
from typing import List, Dict

def search_yelp_businesses(api_key: str, term: str, location: str, limit: int = 50) -> List[Dict]:
    """
    Search for businesses on Yelp
    """
    url = "https://api.yelp.com/v3/businesses/search"
    headers = {"Authorization": f"Bearer {api_key}"}
    
    all_businesses = []
    offset = 0
    max_offset = 1000  # Yelp API limit
    
    while offset < max_offset:
        params = {
            "term": term,
            "location": location,
            "limit": limit,
            "offset": offset
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status()
            data = response.json()
            
            businesses = data.get('businesses', [])
            if not businesses:
                break
                
            all_businesses.extend(businesses)
            print(f"  📥 Fetched {len(businesses)} businesses (Total: {len(all_businesses)})")
            
            if len(businesses) < limit:
                break
                
            offset += limit
            time.sleep(0.2)  # Rate limiting
            
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  Error: {e}")
            break
    
    return all_businesses

def extract_business_info(businesses: List[Dict]) -> pd.DataFrame:
    """Convert Yelp API response to DataFrame"""
    data = []
    
    for biz in businesses:
        data.append({
            'id': biz.get('id'),
            'name': biz.get('name'),
            'rating': biz.get('rating'),
            'review_count': biz.get('review_count'),
            'price': biz.get('price', 'N/A'),
            'latitude': biz.get('coordinates', {}).get('latitude'),
            'longitude': biz.get('coordinates', {}).get('longitude'),
            'address': biz.get('location', {}).get('address1'),
            'city': biz.get('location', {}).get('city'),
            'zip_code': biz.get('location', {}).get('zip_code'),
            'categories': ', '.join([cat['title'] for cat in biz.get('categories', [])]),
        })
    
    return pd.DataFrame(data)

print("✅ Yelp API functions defined!")

## Step 5: Collect Data from Yelp (This takes 2-3 minutes)

In [ ]:
# Categories to search for
SEARCH_CATEGORIES = {
    'coffee': 'Coffee & Tea',
    'breweries': 'Breweries',
    'vegan': 'Vegan Restaurants',
    'yoga': 'Yoga',
    'organic': 'Organic Stores'
}

# Collect data
all_yelp_data = {}

if YELP_API_KEY:
    print(f"🔍 Searching Yelp in {CITY_NAME}...\n")
    
    for term, description in SEARCH_CATEGORIES.items():
        print(f"📍 Searching for: {description}")
        businesses = search_yelp_businesses(
            api_key=YELP_API_KEY,
            term=term,
            location=CITY_NAME,
            limit=50
        )
        
        df = extract_business_info(businesses)
        all_yelp_data[term] = df
        print(f"  ✅ Found {len(df)} locations\n")
    
    # Combine all categories
    yelp_combined = pd.concat([
        df.assign(category=cat) for cat, df in all_yelp_data.items()
    ], ignore_index=True)
    
    print(f"\n📊 Total businesses collected: {len(yelp_combined)}")
    print(f"📍 Unique locations: {yelp_combined['id'].nunique()}")
    
else:
    print("❌ Skipping - API key not configured")
    yelp_combined = pd.DataFrame()

## Step 6: Preview the Data

In [ ]:
if not yelp_combined.empty:
    print("🔍 Sample of collected data:\n")
    display(yelp_combined.head(10))
    
    print("\n📊 Category breakdown:")
    print(yelp_combined['category'].value_counts())
    
    print("\n⭐ Rating statistics:")
    print(yelp_combined['rating'].describe())

## Step 7: Save Data to CSV

In [ ]:
# Create data directory
data_dir = Path("../data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

if not yelp_combined.empty:
    timestamp = datetime.now().strftime('%Y%m%d')
    filename = f"yelp_austin_{timestamp}.csv"
    filepath = data_dir / filename
    
    yelp_combined.to_csv(filepath, index=False)
    print(f"✅ Saved: {filepath}")
    print(f"   📊 {len(yelp_combined)} rows × {len(yelp_combined.columns)} columns")
    print(f"   💾 File size: {filepath.stat().st_size / 1024:.2f} KB")
    print("\n🎉 Data collection complete!")
else:
    print("⚠️  No data to save")

## 🎉 Phase 1 Complete!

You've successfully collected lifestyle amenity data for Austin!

### Next Steps:
1. Review the CSV file in `data/raw/`
2. Let me know when you're ready for Phase 2: Data Cleaning

Great job! 🚀